# Phase 06B.00 — Taxonomy Slice Analysis

Validate the frozen human taxonomy, analyze all nine existing Phase 06A arms, and produce evidence for choosing one novelty track.

**Immutable gates:** `L32-F1`; 298 frozen validation samples; train-side checkpoint selection only; no public-test access. Missing human/input artifacts produce an explicit status and stop—no synthetic labels or provenance.

## 1. Baseline and taxonomy gate

In [ ]:
from pathlib import Path
import json, sys
import pandas as pd
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir()), None)
if ROOT is None: raise RuntimeError("Run inside RoadBuddy")
sys.path.insert(0, str(ROOT / "src")) if str(ROOT / "src") not in sys.path else None
def write_json(path, payload):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    return path
from phase06a_common import sha256_file, sha256_json
from phase06b_common import (LOCKED_BASELINE_WINNER, build_novelty_decision_evidence,
 load_phase06a_predictions, taxonomy_slice_metrics, validate_locked_baseline, validate_taxonomy_frame)
OUT=ROOT/"outputs/phase06b/taxonomy_slice_analysis"; OUT.mkdir(parents=True,exist_ok=True)
TD=ROOT/"outputs/phase06a/question_taxonomy"; TP=TD/"taxonomy_frozen.csv"; TM=TD/"taxonomy_manifest.json"
IDS=ROOT/"data/splits/phase01/validation_sample_ids.json"
baseline_manifest=validate_locked_baseline(ROOT)

In [ ]:
missing=[str(p.relative_to(ROOT)) for p in [TP,TM] if not p.is_file()]
if missing:
 write_json(OUT/"PHASE06B_00_STATUS.json", {"status":"awaiting_annotation","missing":missing,
  "required":["annotator_1.csv","annotator_2.csv","taxonomy_adjudicated.csv","taxonomy_frozen.csv","taxonomy_manifest.json"]})
 raise RuntimeError("Complete independent annotation and adjudication first")
manifest=json.loads(TM.read_text());
if manifest.get("status")!="complete" or manifest.get("taxonomy_sha256")!=sha256_file(TP):
 raise ValueError("Taxonomy manifest is incomplete or hash-mismatched")
ids=json.loads(IDS.read_text()); taxonomy=validate_taxonomy_frame(pd.read_csv(TP),ids)

## 2. Existing-arm slice analysis (no retraining)

In [ ]:
preds=load_phase06a_predictions(ROOT,ids)
slices=taxonomy_slice_metrics(preds,taxonomy,min_rows=30,min_groups=15)
slices.to_csv(OUT/"taxonomy_slice_metrics.csv",index=False)
evidence=build_novelty_decision_evidence(preds[LOCKED_BASELINE_WINNER],taxonomy,min_rows=30,min_groups=15,min_error_share_gap=.10)
write_json(OUT/"novelty_track_evidence.json",evidence)
write_json(OUT/"PHASE06B_00_STATUS.json",{"status":"complete","taxonomy_sha256":sha256_file(TP),
 "evidence_sha256":sha256_json(evidence),"recommendation":evidence["recommendation"],
 "note":"Diagnostic only; Phase06B_01 requires a human decision."})
slices.sort_values(["slice","accuracy"],ascending=[True,False]).head(20)